# Match images to captions with a tiny CLIP

CLIP learns one shared space for images and text by asking, for a batch of pairs, which image
goes with which caption. Here you build that in NumPy on toy features: two linear projections,
L2 normalization, the similarity matrix, the symmetric InfoNCE loss with a temperature, and its
gradient. After a few hundred gradient steps the matched pairs light up the diagonal, and the
model classifies images it has never seen by comparing them with text prompts.

**How to work through it.** Each part asks for one function or class. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

classes = ["cat", "dog", "car", "tree"]
n_classes, latent = len(classes), 32
d_img, d_txt, d_shared = 48, 24, 16    # image feature width, text feature width, shared space width
N = 32                                 # pairs in the training batch

# Each class has a hidden meaning vector. A pair's meaning is its class's vector plus some
# variation. An image encoder and a text encoder each see that meaning through their own random
# linear map, plus noise. The two feature spaces have different widths and unrelated axes, so
# nothing lines up until the projections are trained.
prototypes = rng.normal(size=(n_classes, latent))
A_img = rng.normal(size=(latent, d_img))
A_txt = rng.normal(size=(latent, d_txt))


def make_pairs(n, rng):
    """n (image feature, text feature) pairs. Caption i describes image i. Returns (F, G, labels)."""
    labels = rng.integers(0, n_classes, size=n)
    meaning = prototypes[labels] + 0.5 * rng.normal(size=(n, latent))
    F = meaning @ A_img + 0.5 * rng.normal(size=(n, d_img))    # what the image encoder outputs
    G = meaning @ A_txt + 0.5 * rng.normal(size=(n, d_txt))    # what the text encoder outputs
    return F, G, labels


F_train, G_train, y_train = make_pairs(N, rng)
F_test, G_test, y_test = make_pairs(40, rng)
G_prompts = prototypes @ A_txt     # text features of "a photo of a cat", "a photo of a dog", ...

# The two projections, random to start. They are the only weights that learn.
W_img = rng.normal(size=(d_img, d_shared)) / np.sqrt(d_img)
W_txt = rng.normal(size=(d_txt, d_shared)) / np.sqrt(d_txt)
tau = 0.07


def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


print("image features", F_train.shape, " text features", G_train.shape)
print("classes of the first 10 pairs:", [classes[c] for c in y_train[:10]])

## Part 1 (coding task): Project and normalize

The image encoder's features have 48 numbers and the text encoder's have 24, on unrelated
axes. A linear projection on each side maps them into one shared space of 16 numbers. Then
every vector is scaled to length 1, so a dot product between an image vector and a text
vector is their cosine similarity, a number between -1 and 1.

Write `embed(X, W)`. It returns the rows of `X @ W`, each divided by its own length.
`np.linalg.norm(Z, axis=1, keepdims=True)` gives the lengths as a column.

In [ ]:
def embed(X, W):
    """Project the rows of X with W, then scale every row to length 1."""
    # TODO
    raise NotImplementedError

In [ ]:
I = embed(F_train, W_img)
Tm = embed(G_train, W_txt)
assert I.shape == (N, d_shared) and Tm.shape == (N, d_shared), f"expected {(N, d_shared)} on both sides, got {I.shape} and {Tm.shape}"
assert np.allclose(np.linalg.norm(I, axis=1), 1) and np.allclose(np.linalg.norm(Tm, axis=1), 1), "every row should have length 1"
Z = F_train @ W_img
assert np.allclose(I * np.linalg.norm(Z, axis=1, keepdims=True), Z), "a row should point the same way as its projection"
assert np.allclose(embed(3.0 * F_train, W_img), I), "scaling a feature vector must not change its embedding"
dots = I @ Tm.T
assert dots.min() >= -1 - 1e-9 and dots.max() <= 1 + 1e-9, "dot products of unit vectors are cosines, between -1 and 1"
print("Part 1 passed")

## Part 2 (coding task): The similarity matrix

For a batch of `N` pairs, compare every image with every caption. Entry `[i, j]` of the
similarity matrix is the cosine between image `i` and caption `j`, divided by the
temperature `tau`. The matched pairs are the diagonal. Dividing by a small `tau`
stretches the gaps between the entries, which sharpens the softmaxes of the next part.

Write `similarity(I, Tm, tau)`. It returns the `N x N` matrix `I @ Tm.T / tau`.

In [ ]:
def similarity(I, Tm, tau):
    """N x N matrix: entry [i, j] is the cosine between image i and caption j, divided by tau."""
    # TODO
    raise NotImplementedError

In [ ]:
S = similarity(I, Tm, tau)
assert S.shape == (N, N), f"expected {(N, N)}, got {S.shape}"
assert np.allclose(S[3, 5], I[3] @ Tm[5] / tau), "entry [3, 5] is image 3 against caption 5, divided by tau"
assert np.allclose(similarity(I, Tm, 1.0) / tau, S), "a smaller tau scales every entry up"
assert np.allclose(similarity(I, I, 1.0).diagonal(), 1), "a vector against itself has cosine 1"
assert np.allclose(similarity(Tm, I, tau), S.T), "swapping the sides transposes the matrix"
plt.imshow(S, cmap="gray_r")
plt.title("similarity before training (rows: images, columns: captions)")
plt.colorbar()
plt.show()
print("Part 2 passed")

## Part 3 (coding task): The symmetric InfoNCE loss

InfoNCE is cross-entropy where the classes are the items in the batch. Along row `i` of
the similarity matrix, a softmax over the `N` captions should put its weight on caption
`i`, and the row's loss is `-ln` of that weight. CLIP also runs the same thing down each
column, so each caption picks its image. The CLIP loss is the mean row loss and the mean
column loss, averaged. A model that has learned nothing and spreads its
probability evenly scores `ln N`, and the asserts below force that case with flat
matrices. A real untrained network is sharper than uniform, so its loss starts
above `ln N` and the plot will show it falling past the chance line.

Write `clip_loss(S)`. Take the log of the softmax along each row (subtract the row max,
then subtract the log of the sum of exponentials), read the diagonal, and average its
negative. Do the same along each column, which is the row version applied to `S.T`.
Return the mean of the two.

In [ ]:
def clip_loss(S):
    """Symmetric InfoNCE: cross-entropy with the diagonal as the label, along rows and along columns, averaged."""
    # TODO
    raise NotImplementedError

In [ ]:
# The lesson's batch: 8 pairs from seed 0 cost 0.048065 along rows, 0.065131 along columns, 0.056598 together.
lesson_rng = np.random.default_rng(0)
img8 = lesson_rng.normal(size=(8, 16))
img8 /= np.linalg.norm(img8, axis=1, keepdims=True)
txt8 = img8 + 0.3 * lesson_rng.normal(size=(8, 16))
txt8 /= np.linalg.norm(txt8, axis=1, keepdims=True)
loss8 = clip_loss(img8 @ txt8.T / 0.07)
assert abs(loss8 - 0.056598) < 1e-5, f"the lesson's batch should cost 0.056598, got {loss8:.6f}: average the row and column directions"
assert abs(clip_loss(np.zeros((N, N))) - np.log(N)) < 1e-9, "a constant matrix costs ln N, the chance level"
assert abs(clip_loss(np.full((N, N), 7.0)) - np.log(N)) < 1e-9, "every entry the same means uniform softmaxes, which is a collapsed encoder"
assert clip_loss(100 * np.eye(N)) < 1e-9, "a huge diagonal costs nothing"
assert np.isfinite(clip_loss(1e4 * np.eye(N) - 5e3)), "subtract the max before exponentiating"
S = similarity(embed(F_train, W_img), embed(G_train, W_txt), tau)
print(f"untrained loss {clip_loss(S):.3f}, chance level ln {N} = {np.log(N):.3f}")
print("the same untrained embeddings at several temperatures:")
for t in (1.0, 0.5, 0.2, 0.07, 0.01):
    print(f"  tau {t:<5} loss {clip_loss(similarity(I, Tm, t)):.3f}")
print("Part 3 passed")

## Part 4 (coding task): The gradient of the loss

Training needs the gradient of the loss with respect to the similarity matrix. For
cross-entropy, the gradient with respect to the logits is the softmax minus the one-hot
target, divided by the number of rows. For the row direction that is the row softmax of
`S` minus the identity matrix, divided by `N`. The column direction is the same formula
on `S.T`, transposed back. The CLIP loss averages the two directions, so its gradient
averages the two gradients.

Write `dloss_dS(S)`. It returns an `N x N` matrix. The check compares it with finite
differences. The chain from `S` back to the two projections, through the normalization,
is written out for you in the next part.

In [ ]:
def dloss_dS(S):
    """Gradient of clip_loss(S) with respect to every entry of S."""
    # TODO
    raise NotImplementedError

In [ ]:
S = similarity(embed(F_train, W_img), embed(G_train, W_txt), tau)
dS = dloss_dS(S)
assert dS.shape == (N, N), f"expected {(N, N)}, got {dS.shape}"
eps = 1e-6
for i, j in [(0, 0), (2, 7), (7, 2), (N - 1, 3)]:
    S_plus, S_minus = S.copy(), S.copy()
    S_plus[i, j] += eps
    S_minus[i, j] -= eps
    numeric = (clip_loss(S_plus) - clip_loss(S_minus)) / (2 * eps)
    assert abs(numeric - dS[i, j]) < 1e-7, f"dS[{i}, {j}] should be {numeric:.7f}, got {dS[i, j]:.7f}"
assert abs(dS.sum()) < 1e-12, "the gradient sums to 0: raising every entry by the same amount changes nothing"
assert (np.diag(dS) < 0).all(), "the loss wants every diagonal entry higher"
print("Part 4 passed")

## Part 5 (coding task): Train the two projections

The gradient has to travel from `S` back to `W_img` and `W_txt`: through the dot
products, through the normalization and through the projections. That chain is written
out in `projection_gradients` below. The function you write runs gradient descent with
it.

Write `train(steps, lr)`. At each step, embed both sides, compute `S` and the loss, get
the two gradients from `projection_gradients`, move `W_img` and `W_txt` against them, and
record the loss. Change the arrays in place, with `W_img[:] -= lr * dW_img`,
because inside a function `W_img -= ...` makes `W_img` a local name for the
whole body, so it raises UnboundLocalError before it ever reads the global
array. Return the list of
losses.

In [ ]:
def projection_gradients(F, G, W_img, W_txt, tau):
    """Gradients of clip_loss with respect to W_img and W_txt, by the chain rule."""
    Z_img, Z_txt = F @ W_img, G @ W_txt
    I, Tm = embed(F, W_img), embed(G, W_txt)
    dS = dloss_dS(similarity(I, Tm, tau))
    dI, dTm = dS @ Tm / tau, dS.T @ I / tau          # through S = I Tm^T / tau
    # Through u = z / |z|: the gradient for z is (du - u (u . du)) / |z|.
    dZ_img = (dI - I * (I * dI).sum(axis=1, keepdims=True)) / np.linalg.norm(Z_img, axis=1, keepdims=True)
    dZ_txt = (dTm - Tm * (Tm * dTm).sum(axis=1, keepdims=True)) / np.linalg.norm(Z_txt, axis=1, keepdims=True)
    return F.T @ dZ_img, G.T @ dZ_txt                # through Z = X W


def train(steps, lr):
    """Gradient descent on W_img and W_txt, in place. Returns the loss at each step."""
    # TODO
    raise NotImplementedError

In [ ]:
# First the written-out chain against finite differences, so nothing in it is taken on trust.
dW_img, dW_txt = projection_gradients(F_train, G_train, W_img, W_txt, tau)

def loss_now():
    return clip_loss(similarity(embed(F_train, W_img), embed(G_train, W_txt), tau))

eps = 1e-6
for W, dW, name in [(W_img, dW_img, "W_img"), (W_txt, dW_txt, "W_txt")]:
    for i, j in [(0, 0), (5, 3)]:
        W[i, j] += eps
        up = loss_now()
        W[i, j] -= 2 * eps
        down = loss_now()
        W[i, j] += eps
        numeric = (up - down) / (2 * eps)
        assert abs(numeric - dW[i, j]) < 1e-5, f"d{name}[{i}, {j}] should be {numeric:.6f}, got {dW[i, j]:.6f}"

W_img0, W_txt0 = W_img.copy(), W_txt.copy()
losses = train(200, 0.05)
assert len(losses) == 200, "one loss per step"
assert not np.allclose(W_img, W_img0) and not np.allclose(W_txt, W_txt0), "train must change W_img and W_txt in place"
print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f}, chance level {np.log(N):.3f}")
assert losses[-1] < 1.0, f"after 200 steps the loss should be under 1, got {losses[-1]:.3f}"
S_after = similarity(embed(F_train, W_img), embed(G_train, W_txt), tau)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
axes[0].plot(losses)
axes[0].axhline(np.log(N), color="gray", linestyle="--", label="ln N")
axes[0].set_xlabel("step")
axes[0].set_ylabel("CLIP loss")
axes[0].legend()
axes[1].imshow(S_after, cmap="gray_r")
axes[1].set_title("similarity after training")
plt.tight_layout()
plt.show()
hits = (S_after.argmax(axis=1) == np.arange(N)).mean()
print(f"images whose own caption is their best match: {hits:.0%}")
assert hits >= 0.9, "nearly every image should now pick its own caption"
print("Part 5 passed")

## Part 6 (coding task): Zero-shot classification

Because images and text share one space, classifying an image needs no trained
classifier. Embed the image. Embed one prompt per class, here the text features of "a
photo of a cat", "a photo of a dog" and so on, which sit in `G_prompts`. The prediction
is the class whose prompt is most similar to the image. Nothing was ever trained on class
labels: they were only used to make the toy data.

Write `zero_shot(F, W_img, W_txt)`. It returns one predicted class index per row of `F`,
the index of the prompt with the highest cosine similarity.

In [ ]:
def zero_shot(F, W_img, W_txt):
    """Class index of the most similar prompt in G_prompts, one per image in F."""
    # TODO
    raise NotImplementedError

In [ ]:
pred = zero_shot(F_test, W_img, W_txt)
assert pred.shape == (len(F_test),), f"one prediction per image, got shape {pred.shape}"
assert set(np.unique(pred)) <= set(range(n_classes)), "predictions are class indices"
sims = embed(F_test, W_img) @ embed(G_prompts, W_txt).T
assert (pred == sims.argmax(axis=1)).all(), "pick the prompt with the highest cosine similarity"
acc_after = (pred == y_test).mean()
acc_before = (zero_shot(F_test, W_img0, W_txt0) == y_test).mean()
print(f"zero-shot accuracy on {len(F_test)} unseen images: {acc_before:.0%} with the random projections, {acc_after:.0%} after training")
assert acc_after >= 0.9, "the trained projections should classify nearly every unseen image"
for i in range(5):
    print(f"  image {i}: {classes[y_test[i]]:5s} -> {classes[pred[i]]}")
print("Part 6 passed")

## Solution

### Part 1: Project and normalize

In [ ]:
def embed(X, W):
    """Project the rows of X with W, then scale every row to length 1."""
    Z = X @ W
    return Z / np.linalg.norm(Z, axis=1, keepdims=True)

### Part 2: The similarity matrix

In [ ]:
def similarity(I, Tm, tau):
    """N x N matrix: entry [i, j] is the cosine between image i and caption j, divided by tau."""
    return I @ Tm.T / tau

### Part 3: The symmetric InfoNCE loss

In [ ]:
def clip_loss(S):
    """Symmetric InfoNCE: cross-entropy with the diagonal as the label, along rows and along columns, averaged."""
    def rows_loss(M):
        m = M.max(axis=1, keepdims=True)
        logp = M - m - np.log(np.exp(M - m).sum(axis=1, keepdims=True))
        return -np.mean(np.diag(logp))
    return (rows_loss(S) + rows_loss(S.T)) / 2

### Part 4: The gradient of the loss

In [ ]:
def dloss_dS(S):
    """Gradient of clip_loss(S) with respect to every entry of S."""
    n = len(S)
    rows = (softmax(S, axis=1) - np.eye(n)) / n
    cols = (softmax(S.T, axis=1) - np.eye(n)).T / n
    return (rows + cols) / 2

### Part 5: Train the two projections

In [ ]:
def projection_gradients(F, G, W_img, W_txt, tau):
    """Gradients of clip_loss with respect to W_img and W_txt, by the chain rule."""
    Z_img, Z_txt = F @ W_img, G @ W_txt
    I, Tm = embed(F, W_img), embed(G, W_txt)
    dS = dloss_dS(similarity(I, Tm, tau))
    dI, dTm = dS @ Tm / tau, dS.T @ I / tau          # through S = I Tm^T / tau
    # Through u = z / |z|: the gradient for z is (du - u (u . du)) / |z|.
    dZ_img = (dI - I * (I * dI).sum(axis=1, keepdims=True)) / np.linalg.norm(Z_img, axis=1, keepdims=True)
    dZ_txt = (dTm - Tm * (Tm * dTm).sum(axis=1, keepdims=True)) / np.linalg.norm(Z_txt, axis=1, keepdims=True)
    return F.T @ dZ_img, G.T @ dZ_txt                # through Z = X W


def train(steps, lr):
    """Gradient descent on W_img and W_txt, in place. Returns the loss at each step."""
    losses = []
    for step in range(steps):
        S = similarity(embed(F_train, W_img), embed(G_train, W_txt), tau)
        losses.append(clip_loss(S))
        dW_img, dW_txt = projection_gradients(F_train, G_train, W_img, W_txt, tau)
        W_img[:] -= lr * dW_img
        W_txt[:] -= lr * dW_txt
    return losses

### Part 6: Zero-shot classification

In [ ]:
def zero_shot(F, W_img, W_txt):
    """Class index of the most similar prompt in G_prompts, one per image in F."""
    sims = embed(F, W_img) @ embed(G_prompts, W_txt).T
    return sims.argmax(axis=1)